In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".."))
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(os.path.abspath("."))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

In [2]:
SMOKE_DATASET_DIR = (PROJECT_ROOT / "data" / "smoke_processed").resolve()
SMOKE_MODEL_DIR = (PROJECT_ROOT / "models" / "smoke").resolve()
SMOKE_RESULTS_DIR = (PROJECT_ROOT / "results" / "smoke").resolve()

# 00 Quickstart Smoke Run

This is the fastest notebook entry point in the repository.
It uses the tiny synthetic smoke dataset under `data/smoke_processed/`,
retrains the FiLM autoencoder, and writes evaluation outputs under `results/smoke/`.

In [3]:
import pandas as pd

from src.dataset import load_dataset
from src.config import EvalConfig, TrainConfig
from src.evaluate import evaluate_model
from src.train import train_model
from src.utils import read_json

## Confirm The Small Smoke Dataset

In [4]:
smoke_bundle = load_dataset(SMOKE_DATASET_DIR)
{
    "X_shape": smoke_bundle["X"].shape,
    "C_shape": smoke_bundle["C"].shape,
    "split_counts": smoke_bundle["metadata"]["split"].value_counts().sort_index().to_dict(),
    "feature_columns": smoke_bundle["feature_meta"]["feature_columns"],
}

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\kaspe\\Desktop\\Project\\project\\data\\smoke_processed\\X_all.npy'

## Train A Notebook-Friendly Baseline

In [ ]:
train_cfg = TrainConfig(
    dataset_dir=SMOKE_DATASET_DIR,
    model_dir=SMOKE_MODEL_DIR,
    epochs=8,
    batch_size=4,
    patience=4,
    units=8,
    latent=4,
    device="cpu",
)
train_summary = train_model(train_cfg)
train_summary["detector_meta"]

In [ ]:
history = pd.read_csv(SMOKE_MODEL_DIR / "training_history.csv")
history

## Evaluate The Trained Smoke Model

In [ ]:
eval_cfg = EvalConfig(
    dataset_dir=SMOKE_DATASET_DIR,
    model_dir=SMOKE_MODEL_DIR,
    output_dir=SMOKE_RESULTS_DIR,
    split="test",
    top_k_features=3,
    use_synthetic_injection=False,
    include_gpt_in_stream=True,
    telegram_enabled=True,
    device="cpu",
)
eval_summary = evaluate_model(eval_cfg)
eval_summary["evaluation_summary"]

In [ ]:
predictions = pd.read_csv(SMOKE_RESULTS_DIR / "window_level_predictions.csv")
predictions[
    [
        "window_id",
        "container_id",
        "anomaly_score",
        "top_k_features",
        "predicted_label",
    ]
].head()

In [ ]:
read_json(SMOKE_RESULTS_DIR / "evaluation_summary.json")